# Roadmap Agente RAG Bancario

Notebook guía para construir el prototipo del agente RAG .

**Caso:** agente RAG para consultar documentos internos de subgerencias del banco, identificar hitos, próximos pasos, responsables y cambios organizacionales.

Completa cada sección en orden. Las celdas marcadas como `TODO` son las que debes adaptar a tus documentos, APIs y entorno.

## Paso 0 — Preparación del entorno

**Qué hacer:** instala librerías, configura API keys y define carpetas del proyecto.

**Resultado esperado:** ambiente listo para cargar documentos y crear embeddings.

In [2]:
# Ejecutar una vez si estás en Colab/Jupyter
!pip install -q \
  langchain \
  langchain-openai \
  langchain-community \
  chromadb \
  pypdf \
  python-docx \
  sqlalchemy \
  pandas \
  langchain-text-splitters \
  langchain langchain-openai langchain-community


import os
from pathlib import Path
from google.colab import userdata

PROJECT_DIR = Path.cwd()
DOCS_DIR = PROJECT_DIR / "docs"
DB_DIR = PROJECT_DIR / "vector_db"
OUTPUT_DIR = PROJECT_DIR / "outputs"

DOCS_DIR.mkdir(exist_ok=True)
DB_DIR.mkdir(exist_ok=True)
OUTPUT_DIR.mkdir(exist_ok=True)

# API KEY desde secretos de Colab
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

print("Carpetas listas:", DOCS_DIR, DB_DIR, OUTPUT_DIR)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Falta configurar OPENAI_API_KEY.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 627.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 37.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.8/343.8 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 43.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/

## Paso 1 — Documentos de negocio

**Qué hacer:** carga 3 a 5 documentos representativos de las subgerencias en la carpeta `/docs`.

Ejemplos: reportes de subgerencias, minutas de comité, roadmaps internos, comunicados organizacionales y documentos de cambios estructurales.

**Alcance:** completo, porque el RAG depende directamente de estos documentos.

In [3]:
valid_extensions = [".pdf", ".docx", ".txt", ".md"]

docs = [
    p for p in DOCS_DIR.rglob("*")
    if p.suffix.lower() in valid_extensions
]

print(f"Documentos encontrados: {len(docs)}")

for d in docs:
    print("-", d.relative_to(DOCS_DIR))

Documentos encontrados: 21
- 2024/iniciativas_estrategicas_de_datos_2024.md
- 2024/modernizacion_2024.md
- 2024/procesamiento_de_tarjetas_2024.md
- 2024/data_research_2024.md
- 2024/integracion_2024.md
- 2024/devops_2024.md
- 2024/modelos_y_procesos_de_datos_2024.md
- 2024/00_contexto_organizacional_2024.md
- 2024/digitalizacion_2024.md
- 2024/bigdata_2024.md
- 2026/estructura_organizacional_2026.md
- 2026/00_contexto_organizacional_2026.md
- 2025/iniciativas_estrategicas_de_datos_2025.md
- 2025/modelos_y_procesos_de_datos_2025.md
- 2025/digitalizacion_2025.md
- 2025/00_contexto_organizacional_2025.md
- 2025/procesamiento_de_tarjetas_2025.md
- 2025/integracion_2025.md
- 2025/bigdata_2025.md
- 2025/modernizacion_2025.md
- 2025/devops_2025.md


## Paso 2 — Carga y lectura de documentos

**Qué hacer:** leer los documentos y transformarlos en objetos procesables.

**Resultado esperado:** lista de documentos con texto y metadata de fuente.

In [4]:
from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader
from pathlib import Path

def load_document(path: Path):
    suffix = path.suffix.lower()

    if suffix == ".pdf":
        return PyPDFLoader(str(path)).load()

    if suffix == ".docx":
        return Docx2txtLoader(str(path)).load()

    if suffix in [".txt", ".md"]:
        return TextLoader(str(path), encoding="utf-8").load()

    return []

raw_documents = []

for file_path in docs:
    loaded = load_document(file_path)

    for doc in loaded:
        doc.metadata["source_file"] = file_path.name
        doc.metadata["source_path"] = str(file_path.relative_to(DOCS_DIR))
        doc.metadata["business_domain"] = "subgerencias_banco"

        # Metadata útil para RAG temporal
        parts = file_path.relative_to(DOCS_DIR).parts
        if len(parts) > 1:
            doc.metadata["year"] = parts[0]

    raw_documents.extend(loaded)

print(f"Fragmentos/documentos cargados: {len(raw_documents)}")

if raw_documents:
    print(raw_documents[0].metadata)
    print(raw_documents[0].page_content[:500])

/tmp/ipykernel_1777/846806776.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, Docx2txtLoader, TextLoader


Fragmentos/documentos cargados: 21
{'source': '/content/docs/2024/iniciativas_estrategicas_de_datos_2024.md', 'source_file': 'iniciativas_estrategicas_de_datos_2024.md', 'source_path': '2024/iniciativas_estrategicas_de_datos_2024.md', 'business_domain': 'subgerencias_banco', 'year': '2024'}
---
year: 2024
area: "Iniciativas Estratégicas de Datos"
tipo: "gerencia"
tags:
  - datos
  - crm
  - salesforce
  - tdm
  - dataquality
---

# Gerencia Iniciativas Estratégicas de Datos - 2024

## Principales Logros

- Migración y deprecado de Cloudera.
- Proyecto Riesgo Normativo D00 y SDV.
- Proyecto Riesgo Normativo C64 - Fase 1.
- Gestión de implementación de plataforma de datos BCI Miami.
- Desarrollo de campañas WSB al modelo CRM 2.0.
- CRM 2.0: disminución de SLA de incidentes de 2 horas


In [5]:
print("Archivos detectados:", len(docs))
print("Documentos cargados:", len(raw_documents))

loaded_files = {doc.metadata["source_path"] for doc in raw_documents}
expected_files = {str(p.relative_to(DOCS_DIR)) for p in docs}

missing = expected_files - loaded_files

print("Archivos no cargados:")
for m in sorted(missing):
    print("-", m)

Archivos detectados: 21
Documentos cargados: 21
Archivos no cargados:


## Paso 3 — Chunking con overlap

**Qué hacer:** dividir los documentos en fragmentos más pequeños para mejorar la búsqueda semántica.

Recomendación inicial: `chunk_size=1000`, `chunk_overlap=150`.

**Resultado esperado:** chunks listos para embeddings.

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,
    chunk_overlap=200,
    separators=[
        "\n## ",
        "\n### ",
        "\n- ",
        "\n\n",
        "\n",
        ". ",
        " ",
        ""
    ]
)

chunks = text_splitter.split_documents(raw_documents)

print(f"Chunks generados: {len(chunks)}")

if chunks:
    print(chunks[0].metadata)
    print(chunks[0].page_content[:500])

Chunks generados: 33
{'source': '/content/docs/2024/iniciativas_estrategicas_de_datos_2024.md', 'source_file': 'iniciativas_estrategicas_de_datos_2024.md', 'source_path': '2024/iniciativas_estrategicas_de_datos_2024.md', 'business_domain': 'subgerencias_banco', 'year': '2024'}
---
year: 2024
area: "Iniciativas Estratégicas de Datos"
tipo: "gerencia"
tags:
  - datos
  - crm
  - salesforce
  - tdm
  - dataquality
---

# Gerencia Iniciativas Estratégicas de Datos - 2024

## Principales Logros

- Migración y deprecado de Cloudera.
- Proyecto Riesgo Normativo D00 y SDV.
- Proyecto Riesgo Normativo C64 - Fase 1.
- Gestión de implementación de plataforma de datos BCI Miami.
- Desarrollo de campañas WSB al modelo CRM 2.0.
- CRM 2.0: disminución de SLA de incidentes de 2 horas


**Chunk contextual**
Donde cada chunk agrega contexto automáticamente.

In [7]:

for chunk in chunks:
    chunk.page_content = f"""
Documento: {chunk.metadata['source_file']}
Año: {chunk.metadata['year']}

{chunk.page_content}
"""

## Paso 4 — Embeddings + Vector DB

**Qué hacer:** generar embeddings e indexarlos en una base vectorial. Para prototipo se usa Chroma, porque funciona localmente.

**Alcance:** completo.

In [8]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma

# Modelo embeddings OpenAI
embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Crear vector store
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory=str(DB_DIR),
    collection_name="ti_digital_rag"
)

# Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

print("Vector DB creada en:", DB_DIR)
print("Cantidad chunks indexados:", len(chunks))

Vector DB creada en: /content/vector_db
Cantidad chunks indexados: 33


## Paso 5 — Búsqueda semántica KNN

**Qué hacer:** probar el recuperador con preguntas reales de negocio.

**Resultado esperado:** documentos relevantes recuperados para cada pregunta.

In [9]:
query = "¿Qué iniciativas de IA tenía Digitalización en 2025?"

results = retriever.invoke(query)

for i, r in enumerate(results):
    print(f"\nResultado {i+1}")
    print(r.metadata)
    print(r.page_content[:500])


Resultado 1
{'business_domain': 'subgerencias_banco', 'source': '/content/docs/2024/digitalizacion_2024.md', 'source_path': '2024/digitalizacion_2024.md', 'source_file': 'digitalizacion_2024.md', 'year': '2024'}

Documento: digitalizacion_2024.md
Año: 2024

## Visión 2025

- Consolidar estrategia de impacto y valor medible.
- Observabilidad en todas las soluciones.
- Innovación en el centro e investigación aplicada.
- Nuevo proceso de innovación y publicación de artículos científicos.
- Gobernanza tecnológica para extender herramientas fuera de la subgerencia.
- Low Code, IA y automatizaciones para todo TI Digital.
- Implementación de agentes de IA interactuando directamente con sistemas transversales.


Resultado 2
{'source_path': '2025/digitalizacion_2025.md', 'source': '/content/docs/2025/digitalizacion_2025.md', 'year': '2025', 'business_domain': 'subgerencias_banco', 'source_file': 'digitalizacion_2025.md'}

Documento: digitalizacion_2025.md
Año: 2025

---
year: 2025
area: "Digit

## Paso 6 — Agente orquestador

**Qué hacer:** crear el agente principal que recibe la pregunta, recupera contexto y genera una respuesta.

**Rol:** especialista en estructura bancaria, cambios organizacionales, hitos y próximos pasos por subgerencia.

**Alcance:** completo.

In [10]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

prompt = PromptTemplate.from_template("""
Eres un agente orquestador experto en estructura organizacional bancaria.

Usa SOLO el contexto entregado.
Si no hay información suficiente, dilo explícitamente.
Cita las fuentes usando el campo source_path.

Contexto:
{context}

Pregunta:
{question}

Respuesta:
""")

def format_docs(docs):
    return "\n\n".join(
        [
            f"Fuente: {doc.metadata.get('source_path')}\n{doc.page_content}"
            for doc in docs
        ]
    )

def ask_orchestrator(question: str, k: int = 5):
    source_docs = retriever.invoke(question)

    context = format_docs(source_docs)

    final_prompt = prompt.format(
        context=context,
        question=question
    )

    response = llm.invoke(final_prompt)

    return {
        "result": response.content,
        "source_documents": source_docs
    }

In [11]:
question = "¿Qué iniciativas de IA existían en 2025?"

response = ask_orchestrator(question)

print(response["result"])

print("\n=== FUENTES ===")
for doc in response["source_documents"]:
    print("-", doc.metadata.get("source_path"))

En 2025, las iniciativas de IA incluyeron:

1. **Inspector de Código AI**: Implementación de un sistema que permite la automatización de procesos de análisis de código, logrando que los PAP (Procesos de Análisis de Proyectos) sean 100% automáticos y reduciendo el tiempo de ejecución de días a minutos.

2. **Mejora en el robot de suplantación de identidad digital**: Se realizó una mejora crítica en este sistema, que permitió la captura de 288 millones, lo que sugiere un uso de IA para detectar y prevenir fraudes.

3. **Incorporación de automatizaciones con IA en procesos de desarrollo y pruebas**: Se planificó la automatización de procesos internos de la plataforma de integración utilizando inteligencia artificial.

Estas iniciativas reflejan un enfoque en la automatización y mejora de procesos mediante el uso de inteligencia artificial en diferentes áreas de la organización. 

Fuente: 2025/iniciativas_estrategicas_de_datos_2025.md, 2025/digitalizacion_2025.md, 2025/integracion_2025.md.

## Paso 7 — Agentes workers

**Qué hacer:** separar responsabilidades en funciones o módulos.

Propuesta de alcance parcial:
- Worker 1: búsqueda semántica
- Worker 2: resumen y estructuración de hitos
- No se implementa una arquitectura multiagente compleja todavía.

In [12]:
# =========================
# Guardrail dominio
# =========================

def pregunta_fuera_dominio(question: str):

    keywords_dominio = [
        "subgerencia",
        "gerencia",
        "banco",
        "bancario",
        "organizacional",
        "organización",
        "estructura",
        "hitos",
        "logros",
        "responsables",
        "próximos pasos",
        "digitalización",
        "devops",
        "datos",
        "big data",
        "bigdata",
        "integración",
        "modernización",
        "ia",
        "automatización",
        "agentes",
        "canales",
        "pagos",
        "tarjetas",
        "riesgo",
        "crm",
        "salesforce",
        "mcp"
    ]

    q = question.lower()
    return not any(k in q for k in keywords_dominio)


# =========================
# Worker búsqueda semántica
# =========================

def worker_busqueda_semantica(query: str, k: int = 5):

    retriever_local = vectorstore.as_retriever(
        search_type="similarity",
        search_kwargs={"k": k}
    )

    docs_recuperados = retriever_local.invoke(query)

    return docs_recuperados


# =========================
# Worker resumen / orquestador
# =========================

def worker_resumen_hitos(query: str, k: int = 5):

    # 1. Guardrail fuera de dominio
    if pregunta_fuera_dominio(query):
        return {
            "respuesta": (
                "No existe información suficiente en el dominio documental "
                "para responder esta pregunta. El agente solo responde sobre "
                "documentación organizacional bancaria, gerencias, subgerencias, "
                "hitos, próximos pasos, responsables, datos, DevOps, integración, "
                "modernización, digitalización, pagos, tarjetas e iniciativas IA."
            ),
            "fuentes": [],
            "docs_recuperados": []
        }

    # 2. Retrieval
    source_docs = worker_busqueda_semantica(query, k=k)

    if not source_docs:
        return {
            "respuesta": "No se encontraron documentos relevantes para responder la pregunta.",
            "fuentes": [],
            "docs_recuperados": []
        }

    # 3. Construir contexto
    context = "\n\n".join([
        f"""
Fuente: {doc.metadata.get('source_path')}

{doc.page_content}
"""
        for doc in source_docs
    ])

    # 4. Prompt
    final_prompt = f"""
Eres un especialista en estructura organizacional bancaria.

Usa SOLO el contexto entregado.
No inventes información.
Si no existe información suficiente, indícalo explícitamente.

Responde de forma ejecutiva, clara y ordenada.

Contexto:
{context}

Pregunta:
{query}

Respuesta:
"""

    # 5. LLM
    response = llm.invoke(final_prompt)

    # 6. Fuentes únicas
    fuentes = list({
        doc.metadata.get("source_path")
        for doc in source_docs
        if doc.metadata.get("source_path")
    })

    return {
        "respuesta": response.content,
        "fuentes": fuentes,
        "docs_recuperados": source_docs
    }


# =========================
# Test dentro de dominio
# =========================

resultado = worker_resumen_hitos(
    "Resume los próximos pasos por subgerencia."
)

print("=== RESPUESTA ===")
print(resultado["respuesta"])

print("\n=== FUENTES ===")
for fuente in resultado["fuentes"]:
    print("-", fuente)


# =========================
# Test fuera de dominio
# =========================

resultado_fuera = worker_resumen_hitos(
    "¿Cómo cocinar pizza?"
)

print("\n=== TEST FUERA DE DOMINIO ===")
print(resultado_fuera["respuesta"])
print("Fuentes:", resultado_fuera["fuentes"])

=== RESPUESTA ===
### Próximos Pasos por Subgerencia

#### Subgerencia DevOps
- Generación automática de matriz de cobertura de pruebas.
- Mejoras en métricas de seguridad de código mediante prácticas Shift Left.
- Piloto de modernización del proceso de delivery con microservicios.

#### Subgerencia Big Data
- Implementar cero interacción en despliegue de activos.
- Liberar corrección automática para tipos de datos en MCI.
- Liberar MCI para Real Time.
- Pilotar modelo de corrección y optimización de desarrollo para productos de datos.

#### Subgerencia Digitalización
- Extensión de grafos para todo OyT.
- Extensión de ambientes productivos de agentes.
- Generación de capa corporativa MCP.
- Lanzamiento de Robot HFT para trading Mesa FX.
- Lanzamiento de meta simuladores:
  - Viaje.
  - Autos.
  - Estudios.
  - Agente de créditos de consumo.

=== FUENTES ===
- 2025/devops_2025.md
- 2024/modernizacion_2024.md
- 2025/digitalizacion_2025.md
- 2026/estructura_organizacional_2026.md
- 2025/

## Paso 8 — Agente fiscalizador

**Qué hacer:** validar que la respuesta responda la pregunta, cite fuentes, no invente información y no exponga datos sensibles o PII.

**Alcance:** parcial, con validación básica.

In [13]:
def agente_fiscalizador(question: str, answer: str, sources: list):
    issues = []

    if not answer or len(answer.strip()) < 80:
        issues.append("Respuesta demasiado breve o vacía.")

    if not sources:
        issues.append("No se identificaron fuentes documentales.")

    if question.lower().split()[0] not in answer.lower() and len(answer) < 200:
        issues.append("La respuesta podría no estar respondiendo directamente la pregunta original.")

    sensitive_terms = [
        "rut",
        "contraseña",
        "password",
        "clave",
        "cuenta corriente",
        "número de tarjeta",
        "tarjeta de crédito",
        "cvv"
    ]

    found_sensitive = [
        term for term in sensitive_terms
        if term.lower() in answer.lower()
    ]

    if found_sensitive:
        issues.append(f"Posible información sensible detectada: {found_sensitive}")

    corrected = answer if len(issues) == 0 else (
        "La respuesta requiere revisión antes de ser entregada al usuario. "
        "Motivos: " + "; ".join(issues)
    )

    return {
        "ok": len(issues) == 0,
        "issues": issues,
        "corrected": corrected
    }


check = agente_fiscalizador(
    question="Resume los próximos pasos por subgerencia.",
    answer=resultado["respuesta"],
    sources=resultado["fuentes"]
)

check

{'ok': True,
 'issues': [],
 'corrected': '### Próximos Pasos por Subgerencia\n\n#### Subgerencia DevOps\n- Generación automática de matriz de cobertura de pruebas.\n- Mejoras en métricas de seguridad de código mediante prácticas Shift Left.\n- Piloto de modernización del proceso de delivery con microservicios.\n\n#### Subgerencia Big Data\n- Implementar cero interacción en despliegue de activos.\n- Liberar corrección automática para tipos de datos en MCI.\n- Liberar MCI para Real Time.\n- Pilotar modelo de corrección y optimización de desarrollo para productos de datos.\n\n#### Subgerencia Digitalización\n- Extensión de grafos para todo OyT.\n- Extensión de ambientes productivos de agentes.\n- Generación de capa corporativa MCP.\n- Lanzamiento de Robot HFT para trading Mesa FX.\n- Lanzamiento de meta simuladores:\n  - Viaje.\n  - Autos.\n  - Estudios.\n  - Agente de créditos de consumo.'}

## Paso 9 — SQL para registros

**Qué hacer:** registrar interacciones para trazabilidad.

Campos sugeridos: id, timestamp, query, response, sources, latency, feedback.

**Alcance:** parcial, con SQLite local.

In [14]:
import sqlite3
import json
import time
from datetime import datetime

DB_SQL_PATH = OUTPUT_DIR / "interactions.sqlite"

# =========================
# Conexión DB
# =========================

conn = sqlite3.connect(DB_SQL_PATH)
cursor = conn.cursor()

# =========================
# Tabla
# =========================

cursor.execute("""
CREATE TABLE IF NOT EXISTS interactions (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    timestamp TEXT,
    query TEXT,
    response TEXT,
    sources TEXT,
    latency_seconds REAL,
    feedback TEXT,
    fiscalizador_ok INTEGER,
    fiscalizador_issues TEXT
)
""")

conn.commit()

# =========================
# Logger
# =========================

def log_interaction(
    query,
    response,
    sources,
    latency_seconds,
    feedback=None,
    fiscalizador=None
):

    fiscalizador_ok = None
    fiscalizador_issues = None

    if fiscalizador:
        fiscalizador_ok = int(fiscalizador.get("ok", False))
        fiscalizador_issues = json.dumps(
            fiscalizador.get("issues", []),
            ensure_ascii=False
        )

    cursor.execute(
        """
        INSERT INTO interactions
        (
            timestamp,
            query,
            response,
            sources,
            latency_seconds,
            feedback,
            fiscalizador_ok,
            fiscalizador_issues
        )
        VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        """,
        (
            datetime.now().isoformat(),
            query,
            response,
            json.dumps(sources, ensure_ascii=False),
            latency_seconds,
            feedback,
            fiscalizador_ok,
            fiscalizador_issues
        )
    )

    conn.commit()


# =========================
# TEST
# =========================

start = time.time()

q = "¿Qué cambios organizacionales deben comunicarse a los canales?"

r = worker_resumen_hitos(q)

latency = time.time() - start

# Fiscalizador
check = agente_fiscalizador(
    question=q,
    answer=r["respuesta"],
    sources=r["fuentes"]
)

# Registro
log_interaction(
    query=q,
    response=r["respuesta"],
    sources=r["fuentes"],
    latency_seconds=latency,
    fiscalizador=check
)

print("Interacción registrada en:")
print(DB_SQL_PATH)

print("\nLatencia:")
print(round(latency, 2), "segundos")

print("\nFiscalizador:")
print(check)

print(DB_SQL_PATH)
print(DB_SQL_PATH.exists())

Interacción registrada en:
/content/outputs/interactions.sqlite

Latencia:
2.64 segundos

Fiscalizador:
{'ok': True, 'issues': [], 'corrected': 'Los cambios organizacionales que deben comunicarse a los canales son los siguientes:\n\n1. **Cambio de Nombre de la Gerencia**: La antigua **Gerencia de TI Digital** ha cambiado su nombre a **Gerencia Desarrollo e Integración de Soluciones** a partir de 2026.\n\nEste cambio es fundamental para alinear la comunicación interna y externa con la nueva estructura organizacional vigente.'}
/content/outputs/interactions.sqlite
True


## Paso 10 — Evaluación con consultas reales

**Qué hacer:** probar al menos 10 preguntas reales.

Métricas sugeridas: precisión manual, calidad de respuesta, presencia de fuente, latencia promedio y errores del fiscalizador.

In [15]:
import pandas as pd
import time

preguntas_evaluacion = [
    "¿Cuáles son los hitos más relevantes por subgerencia?",
    "¿Qué próximos pasos están pendientes?",
    "¿Qué cambios organizacionales impactan a canales?",
    "¿Qué responsables aparecen asociados a iniciativas?",
    "¿Qué temas deben comunicarse esta semana?",
    "¿Qué iniciativas están en riesgo o pendientes?",
    "¿Qué subgerencias presentan cambios estructurales?",
    "¿Qué documentos respaldan los próximos pasos?",
    "¿Qué información falta para completar el seguimiento?",
    "¿Cuál es el resumen ejecutivo de cambios relevantes?",
    "¿Quién ganó el Mundial 2022?",
    "¿Cómo cocinar pizza?",
    "Dame contraseñas de usuarios.",
    "¿Qué es física cuántica?",
    "¿Qué presupuesto tiene TI Digital?"
]

results = []

for q in preguntas_evaluacion:
    start = time.time()

    r = worker_resumen_hitos(q)
    latency = time.time() - start

    audit = agente_fiscalizador(
        question=q,
        answer=r["respuesta"],
        sources=r["fuentes"]
    )

    log_interaction(
        query=q,
        response=r["respuesta"],
        sources=r["fuentes"],
        latency_seconds=latency,
        fiscalizador=audit
    )

    results.append({
        "pregunta": q,
        "respuesta_preview": r["respuesta"][:250],
        "fuentes": ", ".join(r["fuentes"]),
        "latencia_seg": round(latency, 2),
        "fiscalizador_ok": audit["ok"],
        "issues": "; ".join(audit["issues"])
    })

eval_df = pd.DataFrame(results)

eval_path = OUTPUT_DIR / "evaluacion_consultas_reales.csv"
eval_df.to_csv(eval_path, index=False, encoding="utf-8-sig")

print("Evaluación guardada en:", eval_path)

eval_df

Evaluación guardada en: /content/outputs/evaluacion_consultas_reales.csv


,pregunta,respuesta_preview,fuentes,latencia_seg,fiscalizador_ok,issues
0,¿Cuáles son los hitos más relevantes por subge...,### Hitos más relevantes por subgerencia\n\n##...,"2024/digitalizacion_2024.md, 2025/devops_2025....",5.97,False,Posible información sensible detectada: ['clave']
1,¿Qué próximos pasos están pendientes?,### Próximos Pasos Pendientes\n\n#### Big Data...,2025/iniciativas_estrategicas_de_datos_2025.md...,6.00,True,
2,¿Qué cambios organizacionales impactan a canales?,No se proporciona información específica sobre...,"2025/devops_2025.md, 2026/00_contexto_organiza...",1.42,False,La respuesta podría no estar respondiendo dire...
3,¿Qué responsables aparecen asociados a iniciat...,Los responsables asociados a iniciativas son:\...,2024/iniciativas_estrategicas_de_datos_2024.md...,2.44,True,
4,¿Qué temas deben comunicarse esta semana?,No existe información suficiente en el dominio...,,0.00,False,No se identificaron fuentes documentales.
5,¿Qué iniciativas están en riesgo o pendientes?,Las iniciativas que están en riesgo o pendient...,2024/iniciativas_estrategicas_de_datos_2024.md...,5.89,True,
6,¿Qué subgerencias presentan cambios estructura...,Las subgerencias que presentan cambios estruct...,"2026/00_contexto_organizacional_2026.md, 2024/...",1.26,True,
7,¿Qué documentos respaldan los próximos pasos?,Los próximos pasos están respaldados por los s...,2025/iniciativas_estrategicas_de_datos_2025.md...,4.49,True,
8,¿Qué información falta para completar el segui...,No existe información suficiente en el dominio...,,0.00,False,No se identificaron fuentes documentales.
9,¿Cuál es el resumen ejecutivo de cambios relev...,No existe información suficiente en el dominio...,,0.00,False,No se identificaron fuentes documentales.


In [18]:
import pandas as pd

df_logs = pd.read_sql_query(
    "SELECT * FROM interactions",
    conn
)

df_logs[
    [
        "timestamp",
        "query",
        "latency_seconds",
        "fiscalizador_ok"
    ]
].tail(10)

,timestamp,query,latency_seconds,fiscalizador_ok
6,2026-05-25T23:43:59.084533,¿Qué iniciativas están en riesgo o pendientes?,5.887475,1
7,2026-05-25T23:44:00.351331,¿Qué subgerencias presentan cambios estructura...,1.255294,1
8,2026-05-25T23:44:04.845526,¿Qué documentos respaldan los próximos pasos?,4.485577,1
9,2026-05-25T23:44:04.855330,¿Qué información falta para completar el segui...,0.000040,0
10,2026-05-25T23:44:04.864880,¿Cuál es el resumen ejecutivo de cambios relev...,0.000028,0
11,2026-05-25T23:44:05.837657,¿Quién ganó el Mundial 2022?,0.963722,0
12,2026-05-25T23:44:05.849030,¿Cómo cocinar pizza?,0.000035,0
13,2026-05-25T23:44:05.857859,Dame contraseñas de usuarios.,0.000026,0
14,2026-05-25T23:44:05.865789,¿Qué es física cuántica?,0.000028,0
15,2026-05-25T23:44:05.872944,¿Qué presupuesto tiene TI Digital?,0.000022,0


## Paso 11 — GitHub

**Qué hacer:** ordenar el proyecto para subirlo a GitHub.

Estructura recomendada:
```text
rag-bancario/
  docs/
  src/
  tests/
  outputs/
  requirements.txt
  README.md
  .env.example
  .gitignore
```

**Alcance:** completo.

In [ ]:
requirements = '''langchain
langchain-openai
langchain-community
chromadb
pypdf
python-docx
pandas
sqlalchemy
'''

req_path = PROJECT_DIR / "requirements.txt"
req_path.write_text(requirements, encoding="utf-8")

print("Archivos creados:", req_path)

Archivos creados: /content/requirements.txt /content/README.md


## Paso 12 — Cloud Run y Go Live

**Qué hacer:** dejar explícito que no se implementa en esta etapa.

**Por qué queda fuera del alcance:**
- requiere configuración cloud
- manejo de secretos
- control de costos
- monitoreo productivo
- seguridad enterprise
- validaciones con áreas internas del banco

In [ ]:
fuera_de_alcance = {
    "Cloud Run": "No se implementa en esta etapa. Requiere infraestructura cloud, IAM, secretos y costos.",
    "Deploy & Go Live": "No se implementa en esta etapa. Se entrega prototipo funcional, no solución productiva enterprise."
}

fuera_de_alcance

{'Cloud Run': 'No se implementa en esta etapa. Requiere infraestructura cloud, IAM, secretos y costos.',
 'Deploy & Go Live': 'No se implementa en esta etapa. Se entrega prototipo funcional, no solución productiva enterprise.'}

In [ ]:
!git init
!git config --global user.name "larayad"
!git config --global user.email "luisaraya24@gmail.com"

Reinitialized existing Git repository in /content/.git/


In [ ]:
%%writefile .gitignore
.env
*.sqlite
vector_db/
__pycache__/
.ipynb_checkpoints/

Overwriting .gitignore


In [ ]:
!git add .
!git commit -m "feat: agente RAG bancario con embeddings y evaluación"

[master (root-commit) 974cd2c] feat: agente RAG bancario con embeddings y evaluación
 47 files changed, 52118 insertions(+)
 create mode 100644 .config/.last_opt_in_prompt.yaml
 create mode 100644 .config/.last_survey_prompt.yaml
 create mode 100644 .config/.last_update_check.json
 create mode 100644 .config/active_config
 create mode 100644 .config/config_sentinel
 create mode 100644 .config/configurations/config_default
 create mode 100644 .config/default_configs.db
 create mode 100644 .config/gce
 create mode 100644 .config/hidden_gcloud_config_universe_descriptor_data_cache_configs.db
 create mode 100644 .config/logs/2026.05.21/13.26.07.261908.log
 create mode 100644 .config/logs/2026.05.21/13.26.27.601334.log
 create mode 100644 .config/logs/2026.05.21/13.26.42.337595.log
 create mode 100644 .config/logs/2026.05.21/13.26.44.509737.log
 create mode 100644 .config/logs/2026.05.21/13.26.58.743083.log
 create mode 100644 .config/logs/2026.05.21/13.26.59.793897.log
 create mode 100644 

In [ ]:
from google.colab import userdata

token = userdata.get("gh_token")

In [ ]:
repo_url = f"https://{token}@github.com/larayad/rag-bancario-uai.git"
!git branch -M main
!git branch
!git status
!git log --oneline -5
!git checkout main
!git restore --staged outputs/evaluacion_consultas_fuera_dominio_bancario.csv
!git remote remove origin
!git remote add origin $repo_url
!git push -u origin main --force

* main
On branch main
nothing to commit, working tree clean
974cd2c (HEAD -> main) feat: agente RAG bancario con embeddings y evaluación
Already on 'main'
Enumerating objects: 59, done.
Counting objects: 100% (59/59), done.
Delta compression using up to 2 threads
Compressing objects: 100% (52/52), done.
Writing objects: 100% (59/59), 8.43 MiB | 2.04 MiB/s, done.
Total 59 (delta 6), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (6/6), done.
To https://github.com/larayad/rag-bancario-uai.git
 + 6fa3d0d...974cd2c main -> main (forced update)
Branch 'main' set up to track remote branch 'main' from 'origin'.


In [16]:
!git remote set-url origin https://github.com/larayad/rag-bancario-uai.git

fatal: not a git repository (or any of the parent directories): .git


# Checklist final

Antes de entregar, verifica:

- [ ] Tengo 3 a 5 documentos en `/docs`
- [ ] Los documentos cargan correctamente
- [ ] Se generan chunks
- [ ] Se crea la base vectorial
- [ ] El agente responde preguntas
- [ ] El fiscalizador valida fuentes
- [ ] Se registran interacciones básicas
- [ ] Tengo 10 preguntas de evaluación
- [ ] Tengo README y requirements.txt
- [ ] Tengo claro qué está dentro, parcial y fuera del alcance